## Fine Tuning

In [ ]:
#Open API Fine Tuning Dashboard URL: https://platform.openai.com/finetune/

In [7]:
from dotenv import load_dotenv
from openai import OpenAI
import os

# --- Load API Key ---
load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=my_api_key)

In [ ]:
#Upload training file for fine-tuning to OpenAI
with open("data/training_data.jsonl", "rb") as f:
    training_file_obj = client.files.create(file=f, purpose="fine-tune")

print("Uploaded training file ID:", training_file_obj.id)

#Upload validation file for fine-tuning to OpenAI
with open("data/validation_data.jsonl", "rb") as f:
    validation_file_obj = client.files.create(file=f, purpose="fine-tune")

print("Uploaded validation file ID:", validation_file_obj.id)

In [ ]:
# Create a fine-tuning job
job = client.fine_tuning.jobs.create(
    training_file=training_file_obj.id,       # The file ID you uploaded
    validation_file=validation_file_obj.id,     # Optional
    model="gpt-4.1-nano-2025-04-14",               # or the allowed model you want to fine-tune at https://developers.openai.com/api/docs/guides/model-optimization
    suffix="brand-customer-support"       # Optional model name suffix
)

print("Fine-tune job created:", job.id)

In [1]:
#Get status of fine-tuning job
jobs = client.fine_tuning.jobs.list(limit=5)
for j in jobs.data:
    print(j.id, j.status, j.fine_tuned_model)

NameError: name 'client' is not defined

In [8]:
def ask_question_without_finetuning(prompt):
    print(f"User asked: {prompt}")
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer as concisely as possible."},
            {"role": "user", "content": prompt}
        ]
    )
    print (response)
    return response.choices[0].message.content  

def ask_question_with_finetuning(prompt):
    print(f"User asked: {prompt}")
    response = client.chat.completions.create(
        # model="ADD_YOUR_MODEL_NAME", #IMPORTANT: Use the fine-tuned model name from your fine tuning
        model="ft:gpt-4.1-nano-2025-04-14:personal:brand-customer-support:D4TwzJgJ", #IMPORTANT: Use the fine-tuned model name from your fine tuning
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer as concisely as possible."},
            {"role": "user", "content": prompt}
        ]
    )
    print (response)
    return response.choices[0].message.content  

In [9]:
import time
while True:
    # Ask user for a question
    user_prompt = input("Ask something: ")

    if (user_prompt.lower() != 'quit'):
        # Get and print the response
        response = ask_question_without_finetuning(user_prompt)
        print("\n[WITHOUT FINE TUNING] OpenAI says:", response)

        response = ask_question_with_finetuning(user_prompt)
        print("\n[WITH FINE TUNING] OpenAI says:", response)

        # add delay of 3 seconds
        time.sleep(3)
    else:
        break    

User asked: my computer restarts and heats up
ChatCompletion(id='chatcmpl-DTVrMRto3XzOriEJcN9iksRgdUwVZ', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Your computer restarting and heating up could indicate hardware or software issues. Try these steps:\n\n1. **Check for Overheating:** Ensure fans are working, vents are clear, and dust is cleaned out.\n2. **Monitor Temperatures:** Use software like HWMonitor or Core Temp.\n3. **Update Drivers and BIOS:** Keep your system and drivers current.\n4. **Scan for Malware:** Run antivirus scans.\n5. **Check for Software Issues:** Remove recent updates or software that may cause conflicts.\n6. **Inspect Hardware:** Test RAM and GPU for errors; consider reseating components.\n7. **Power Supply:** Ensure your power supply is adequate and functioning properly.\n8. **Seek Professional Help:** If issues persist, consult a technician.\n\nIf the problem continues, provide details about your system 